# Notebook 04: Evasion — functionality-preserving attacks on the static detector

**Class 04_03.** The PE-header detector of notebook 01 classifies a file from header fields. An attacker
controls most of those fields *without changing what the program does*: timestamps, checksums, linker and OS
versions, section sizes and reserves are metadata. We attack the model with three functionally-preserving
manipulations:

1. **Malleable-field rewrite** — set the 30+ freely-settable fields to the benign median (a template attack).
2. **Benign mimicry** — copy an entire benign profile and keep only a few payload fields.
3. **Greedy black-box** — query the detector, rewrite the one field that lowers the score most, repeat.

Then a **defence**: retraining with attacked samples in the training set.

In [ ]:
import sys

import numpy as np

sys.path[:0] = [".", "..", "../../src", "../../../src"]
import aasplot as ap
import malwlib as mw

ap.setup("04")
X, y, cols = mw.load_pe()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
mal = np.where(yte == 1)[0]
print(f"baseline malicious recall {recall_score(yte, rf.predict(Xte), pos_label=1):.4f}")

med = np.median(Xtr[ytr == 0], 0)  # benign template (median profile)


def report(name, Xa, mask):
    p = rf.predict_proba(Xa[mask])[:, 1]
    ev = float((p < 0.5).mean())
    print(f"{name:38s} evasion {ev:.3f}  F1-mal {f1_score(yte[mask], p > 0.5, pos_label=1):.4f}")
    return ev

## Attack A: malleable fields -> benign medians

`malwlib.MALLEABLE` lists the header fields an attacker may set freely: rewriting them costs nothing at
runtime. On this dataset, that alone is enough to flip almost every malicious file past the detector.

In [ ]:
mal_fields = [cols.index(c) for c in mw.MALLEABLE if c in cols]
print(f"malleable fields present: {len(mal_fields)} of {len(mw.MALLEABLE)}")
XA = mw.median_rewrite(Xte, Xte, mal_fields, med)
evA = report("A malleable rewrite", XA, mal)

## Attack B: benign mimicry (keep k payload fields)

Copy a random benign profile; keep `k` fields of the payload. Small `k` = strong attack, but also more
likely to break the program if a kept field is not actually malleable — the functionality-preservation
trade-off of mimicry attacks.

In [ ]:
rng = np.random.default_rng(0)
evs = []
for k in [40, 20, 10, 5]:
    XB = mw.benign_template_attack(Xte[mal], Xtr[ytr == 0], keep=k, rng=rng, cols=X.shape[1])
    XBf = Xte.copy()
    XBf[mal] = XB
    evs.append(report(f"B mimicry keep={k:2d}", XBf, mal))

In [ ]:
f, ax = ap.fig("col", 1.8)
ax.plot([40, 20, 10, 5], evs, "-o", color=ap.RED)
ax.set_xlabel("payload fields kept k")
ax.set_ylabel("evasion rate")
ax.set_ylim(0, 1.05)
ap.save(f, "d04_mimicry")

## Attack C: greedy black-box rewriting

The attacker knows nothing but the label queries: try each malleable field at its benign median, apply the
one that lowers the score the most, repeat up to 30 steps. This is the realistic attack — no gradients, no
training data.

In [ ]:
sub = mal[:60]
XC = Xte.copy()
for i in sub:
    XC[i] = mw.greedy_evasion(rf, Xte[i], mal_fields, med)
evC = report("C greedy black-box (n=60)", XC, sub)

## Summary and the defence: adversarial retraining

All three attacks fold the detector. Retraining on attacked samples removes the attack it saw (A: evasion 0.000);
guide 04_03 tests the defended model against B and C.

In [ ]:
ev0 = report("baseline (no attack)", Xte, mal)

# defence: add Attack-A samples of the *training* malware to the training set, retrain
Xadv = mw.median_rewrite(Xtr, Xtr, mal_fields, med)
rf_def = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
rf_def.fit(np.vstack([Xtr, Xadv[ytr == 1]]), np.concatenate([ytr, np.ones((ytr == 1).sum(), int)]))
rf, rf_keep = rf_def, rf  # swap: `rf` now points at the defended model inside `report`
evA_def = report("A malleable rewrite (defended)", XA, mal)
XB10 = Xte.copy()
XB10[mal] = mw.benign_template_attack(Xte[mal], Xtr[ytr == 0], keep=10, rng=np.random.default_rng(0), cols=X.shape[1])
evB_def = report("B mimicry keep=10 (defended)", XB10, mal)
XCd = Xte.copy()
for i in sub:
    XCd[i] = mw.greedy_evasion(rf_def, Xte[i], mal_fields, med)
evC_def = report("C greedy black-box (defended)", XCd, sub)
rf = rf_keep
print(
    f"defended model baseline recall {recall_score(yte, rf_def.predict(Xte), pos_label=1):.4f} "
    f"(defended attack-A evasion {evA_def:.3f})"
)

In [ ]:
f, ax = ap.fig("col", 1.9)
names = ["baseline", "A malleable", "B mimicry", "C greedy"]
vals = [ev0, evA, evs[-1], evC]  # undefended model under the three attacks
ax.bar(np.arange(4), vals, 0.55, color=[ap.GREY, ap.RED, ap.RED, ap.RED])
ax.set_xticks(np.arange(4), names, rotation=15)
ax.set_ylabel("evasion rate")
ax.set_ylim(0, 1.05)
ap.save(f, "d04_evasion_summary")

## Takeaways

- Header fields are *metadata*: rewriting the 30+ malleable ones to benign medians gives ~100% evasion with
  zero effect on what the program does — the classic functionality-preserving result (Chen et al. 2017,
  Kolosnjaji et al. 2018, Demetrio et al. 2020).
- Mimicry trades attack strength for preservation: fewer kept payload fields = more evasion, more risk of
  breaking the sample.
- Greedy black-box queries reach the same result without gradients. Retraining with Attack-A samples
  neutralises *that* attack completely (evasion 0, recall unchanged) — but only that attack: an adaptive
  attacker switches to mimicry and the arms race restarts.
- Dynamic sequence models (notebook 03) are the harder target: behaviour must actually be produced.